# 06 · Preprocessing and relabelling of the training data

Builds **one training file** that every model notebook reads: the whole uk dataset, cleaned and deduplicated, plus a
cleaned and deduplicated **ru** sample, sized so that ru is `RU_SHARE` (20%) of the train sentences. Test has 7% Russian
posts and the CV gold has Russian forms (`Фастовском`, `Бучанском`), which the uk data barely covers.

**Output:** `ARTIFACTS/train_v2.parquet` with `text, loc_markers, org_markers, per_markers, is_valid, lang`.
- `is_valid == 0` is train (uk + ru), `is_valid == 1` is `es`: the same 10k uk rows as in every other notebook,
  cleaned the same way as train.
- Markers stay strings in the CSV format (`"[(0, 4)]"`, end exclusive), so `parse_markers` in section 0 reads them.
- **CV (`labeling_sample`) and test are never touched.** Every fix below is a rule on the silver train labels, kept in
  code so it is reproducible (CLAUDE.md, "Silver label noise").

**Steps:** apostrophes and decomposed letters (2.1) → site tails (2.2) → hyphenated names (2.3) → punctuation at span edges (2.4) → shifted
spans (2.5) → LOC on names of people (2.6) → exact duplicates and label conflicts (2.7) → ru boilerplate cap and
sample (2.8) → save (3).

**How to run:** locally, top to bottom (the full ru file is read in chunks, so it needs a few GB of RAM). Then put
`train_v2.parquet` into `MyDrive/dl-a3-artifacts/` for Colab. Never commit it: it is competition data.
`QUICK_RUN=1` builds a small file in `artifacts/quick_run/` to check the code.

In [1]:
NOTEBOOK = "06-preprocessing"

## 0. Setup
- **Colab:** add `KAGGLE_USERNAME`, `KAGGLE_KEY` (and optionally `WANDB_API_KEY`) to Colab Secrets (key icon, with
  notebook access on), or upload `secrets.env` with them to `/content/`. The cell
  loads the keys (it prints only their names), installs missing packages, and downloads the data with the Kaggle CLI.
  The cell mounts Google Drive, and all outputs go to `MyDrive/dl-a3-artifacts`.
- **Locally:** the data is read from `data/raw/` (or `data/` if there is no `raw/`), outputs go to `artifacts/`
  (git-ignored). Packages come from the `uv` environment; nothing is installed from the notebook.

In [2]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

COMPETITION = "kse-ua-location-extraction-2026"
DATA_FILES = ["uk_geo_dataset.csv", "ru_geo_dataset.csv", "labeling_sample.csv", "test.csv"]
IN_COLAB = Path("/content").exists()
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"


SECRET_KEYS = ("KAGGLE_USERNAME", "KAGGLE_KEY", "WANDB_API_KEY")


def load_secrets(path: Path = Path("/content/secrets.env")) -> None:
    """Keys from an uploaded /content/secrets.env, else from Colab Secrets (key icon in the left panel, with
    "Notebook access" on). Kaggle keys are required (data download, submissions); W&B is optional."""
    if path.exists():
        for line in path.read_text().splitlines():
            if "=" in line and not line.lstrip().startswith("#"):
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()
    else:
        from google.colab import userdata

        for key in SECRET_KEYS:
            try:
                os.environ[key] = userdata.get(key)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    print("keys loaded:", [k for k in SECRET_KEYS if os.environ.get(k)])
    assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"), \
        "add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (or upload secrets.env to /content)"


def download_data(raw: Path) -> None:
    raw.mkdir(parents=True, exist_ok=True)
    for name in DATA_FILES:
        if (raw / name).exists():
            continue
        subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION, "-f", name, "-p", str(raw)], check=True)
        archive = raw / f"{name}.zip"
        if archive.exists():  # large files arrive zipped
            shutil.unpack_archive(archive, raw)
            archive.unlink()


def install_missing(packages: dict[str, str]) -> None:
    """{import name: pip name}; installs only what is missing."""
    missing = [pip for module, pip in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    install_missing({"lightning": "lightning", "tokenizers": "tokenizers", "wandb": "wandb", "kaggle": "kaggle"})
    load_secrets()
    RAW = Path("/content/data/raw")
    download_data(RAW)
    ARTIFACTS = Path("/content/drive/MyDrive/dl-a3-artifacts")  # the team's shared folder; mounted above
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RAW = ROOT / "data/raw" if (ROOT / "data/raw").exists() else ROOT / "data"
    ARTIFACTS = ROOT / "artifacts"
if QUICK_RUN:
    ARTIFACTS = ARTIFACTS / "quick_run"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"data: {RAW} | outputs: {ARTIFACTS} | colab: {IN_COLAB} | quick run: {QUICK_RUN}")

data: /home/taras/Documents/Workspace/KSE/deep_learning/lab-3/data | outputs: /home/taras/Documents/Workspace/KSE/deep_learning/lab-3/artifacts | colab: False | quick run: False


### 0.1 Libraries and seed
Data libraries are always needed. `torch` and `lightning` are imported only where they are installed (always on
Colab), so the notebook also runs in a light local environment for data exploration. `SEED = 42` is fixed for
`random`, `numpy` and, when present, `torch` (CPU and CUDA) through `L.seed_everything`.

In [3]:
import ast
import random
import re
import string
import time
import unicodedata
import warnings
from collections import Counter
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HAS_TORCH = importlib.util.find_spec("lightning") is not None
if HAS_TORCH:
    import lightning as L
    import torch
    import torch.nn as nn

    L.seed_everything(SEED, workers=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
else:
    print("torch/lightning not installed: data exploration only")

torch/lightning not installed: data exploration only


## 1. Training data
- **uk** = `uk_geo_dataset` (~1M Ukr-Synth sentences). `is_valid==0` is train, `is_valid==1` is `es` (early
  stopping), the same split as in every other notebook.
- **ru** = `ru_geo_dataset` (~8M nerus sentences). Only a random sample is kept: `RU_OVERSAMPLE` times the final size,
  read chunk by chunk, so the whole 2 GB file never sits in memory. Deduplication, conflicts and the boilerplate cap
  (2.7, 2.8) shrink it, then section 2.8 samples the final `RU_SHARE`.
- `QUICK_RUN` keeps 40k uk train and 2k `es` rows, and reads the first 400k ru rows.

In [4]:
MARKERS = ["loc_markers", "org_markers", "per_markers"]
RU_SHARE = 0.2  # share of ru sentences in the final train
RU_ROWS = 8_028_840  # rows in ru_geo_dataset.csv
RU_OVERSAMPLE = 3  # read 3x the final ru size: dedup, conflicts and the boilerplate cap remove part of it


def load_train_data() -> tuple[pd.DataFrame, pd.DataFrame, int]:
    t0 = time.time()
    uk = pd.read_csv(RAW / "uk_geo_dataset.csv")
    if QUICK_RUN:
        uk = pd.concat([uk[uk.is_valid == 0].sample(40_000, random_state=SEED),
                        uk[uk.is_valid == 1].sample(2_000, random_state=SEED)])
    ru_target = int((uk.is_valid == 0).sum() * RU_SHARE / (1 - RU_SHARE))
    ru_rows = 400_000 if QUICK_RUN else RU_ROWS
    frac = min(1.0, RU_OVERSAMPLE * ru_target / ru_rows)
    chunks = pd.read_csv(RAW / "ru_geo_dataset.csv", usecols=["text", *MARKERS, "doc_id"], chunksize=500_000,
                         nrows=ru_rows if QUICK_RUN else None)
    ru = pd.concat([c.sample(frac=frac, random_state=SEED + i) for i, c in enumerate(chunks)], ignore_index=True)
    ru["is_valid"] = 0
    ru = ru.dropna(subset=["text"])
    print(f"loaded in {time.time() - t0:.0f}s; ru target {ru_target:,} rows, read a {frac:.1%} sample")
    return uk, ru, ru_target


uk_df, ru_df, RU_TARGET = load_train_data()
for name, df in [("uk", uk_df), ("ru", ru_df)]:
    print(f"{name}: {len(df):,} rows, {df.memory_usage(deep=True).sum() / 2**30:.2f} GiB")
print(uk_df.is_valid.value_counts().rename({0: "train", 1: "es"}).to_dict())

loaded in 14s; ru target 250,000 rows, read a 9.3% sample
uk: 1,010,000 rows, 0.23 GiB
ru: 750,006 rows, 0.20 GiB
{'train': 1000000, 'es': 10000}


## 2. Preprocessing (train data only)

### 2.1 Apostrophes and decomposed letters
Ukrainian writes the apostrophe in several ways, and the data mixes them (in-word counts):

| | `'` U+0027 | `’` U+2019 | `ʼ` U+02BC | `‘` U+2018 | `` ` `` U+0060 |
|---|---|---|---|---|---|
| uk | 56,341 | 53,608 | 239 | 1,509 | 1,026 |
| ru | 28,423 | 4,117 | 26 | 6 | 304 |
| CV | 3 | 3 | 1 | 0 | 0 |
| test | 56 | 106 | 35 | 1 | 1 |

Without normalization `Слов'янськ` and `Слов’янськ` are different words with different ids and different fastText
lookups. Every variant becomes `'` (U+0027), the most common one.

- The replacement is **one character for one character**, so every `loc_markers` offset stays valid.
- **CV and test:** normalize only the text the model reads, and cut the output strings from the **original** text.
  Test mostly uses `’` and `ʼ`, and the metric keeps apostrophes, so a normalized `Слов'янськ` would not match a gold
  `Слов’янськ`.

In [5]:
APOSTROPHES = "’ʼ‘`′´"
APOSTROPHE_TABLE = str.maketrans(APOSTROPHES, "'" * len(APOSTROPHES))


def normalize_apostrophes(texts: pd.Series) -> pd.Series:
    """Every apostrophe variant -> ' (U+0027). One char for one char, so span offsets stay valid."""
    return texts.str.translate(APOSTROPHE_TABLE)


assert normalize_apostrophes(pd.Series(["Слов’янськ ʼ`"])).iloc[0] == "Слов'янськ ''"
for df in (uk_df, ru_df):
    df["text"] = normalize_apostrophes(df.text)

**Decomposed letters.** 777 uk texts (and about 60 per 1M ru rows) write `й` as `и` + U+0306 and `ї` as `і` + U+0308,
e.g. `найменше`, `свій`, `річний`, which look the same on screen. `\w` does not match a combining mark, so `TOKEN_RE`
cuts such a word into pieces and the pieces get their own ids. A few texts also carry stress marks (U+0301, `а́кторів`)
or a strikethrough (U+0336).

- Each mark is **composed** into the letter before it (NFC: `и` + U+0306 → `й`) or, if there is no composed letter
  (stress, strikethrough), **dropped**.
- That makes the text shorter, so every `loc`/`org`/`per` offset is moved through a position map. Only rows with a mark
  are touched.
- **Train only.** CV and test have no combining marks, so section 0 needs no change.

In [6]:
def parse_markers(value) -> list[tuple[int, int]]:
    """`loc_markers` as stored in the CSV ("[(0, 4), (10, 15)]") -> [(start, end), ...], end exclusive."""
    if not isinstance(value, str):
        return []
    return [(int(a), int(b)) for a, b in ast.literal_eval(value.strip() or "[]")]


COMBINING_RE = re.compile("[\u0300-\u036f]")


def compose_marks(text: str) -> tuple[str, list[int]]:
    """Fold every combining mark into the letter before it (NFC) or drop it. Returns the new text and `pos`, where
    `pos[i]` is the new offset of old offset `i` (for i in 0..len(text))."""
    out, pos = [], []
    for ch in text:
        pos.append(len(out))
        if COMBINING_RE.match(ch):
            if out and len(c := unicodedata.normalize("NFC", out[-1] + ch)) == 1:
                out[-1] = c
            continue
        out.append(ch)
    pos.append(len(out))
    return "".join(out), pos


def compose_texts(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = df.index[df.text.str.contains(COMBINING_RE)]
    df = df.copy()
    for i in rows:
        text, pos = compose_marks(df.at[i, "text"])
        df.at[i, "text"] = text
        for col in MARKERS:
            df.at[i, col] = str([(pos[a], pos[b]) for a, b in parse_markers(df.at[i, col])])
    print(f"{name}: {len(rows):,} texts with combining marks composed")
    return df


_t, _pos = compose_marks("Киі\u0308в, а\u0301ктор")
assert _t == "Київ, актор" and _t[_pos[0]:_pos[5]] == "Київ" and _pos[-1] == len(_t)
uk_df, ru_df = compose_texts(uk_df, "uk"), compose_texts(ru_df, "ru")

uk: 777 texts with combining marks composed


ru: 21 texts with combining marks composed


### 2.2 News-site tails
About 1.3% of uk texts are scraped headlines with the site's breadcrumb or footer left on. The biggest source is TSN,
`<headline> - <rubric> - TCH.`, e.g. `Психолог Наталія Холоденко вдруге стала мамою - Гламур - TCH.`. The rubric
words are tagged as LOC (`Укрaїнa`, `Світ`, `Київ`, `Сході`) or PER (`Гламур`), so they teach the model to tag a site
menu as a place. Other sites found by counting the most frequent text endings:

| site | tail | texts |
|---|---|---|
| TSN | ` - <rubric> - TCH.` | 9,249 |
| Молодий буковинець | ` » Новини Чернівці: Інформаційний портал «Молодий буковинець»Завантаження.` | 983 |
| Korrespondent | ` - Korrespondent.` | 847 |
| LB.ua | ` - портал новин LB.` | 676 |
| Pravda | ` - Pravda.` | 641 |
| 112 Україна | ` - 112 УкраїнаТок-шоу "Пульс".` | 614 |
| ZIK | ` – новини ZIK.`, ` Почніть свій день з новини Встановити ZIK.` | 531 |
| Україна Молода | ` - - Україна Молода Передплата на “УМ”` | 358 |
| ZAXID | ` - ZAXID.` | 250 |
| 24 Канал | ` - 24 КаналCreated with Sketch.` | 198 |
| Доктор Пі | ` - ПіКСторінка “Доктора Пі”` | 153 |

The tail is cut from the text, and every LOC/ORG/PER span that ends inside it is dropped. The tail is at the end, so
the other offsets don't move. Up to two short rubric segments before the site name are cut too (`- новини України -
Укрaїнa - TCH.`). This runs **before** deduplication, because the same headline can come from two sites.

Applied to the whole uk file (train and `es`), so early stopping also measures the clean task. This is a fix of the
silver training data, not a rule applied at prediction time: CV and test are never touched.

In [7]:
RUBRIC = r"(?:[^-–—\n]|(?<=\w)-(?=\w)){1,30}"  # one breadcrumb segment: no spaced dash, up to 30 chars
DASH_SITES = [r"TCH\.", r"Korrespondent\.", r"портал новин LB\.", r"Pravda\.", r"новини ZIK\.", r"ZAXID\.",
              r'112 УкраїнаТок-шоу "Пульс"\.', r"Україна Молода Передплата на “УМ”", r"ПіКСторінка “Доктора Пі”",
              r"(?:Телеканал новин 24|24 Канал)Created with Sketch\."]
SITE_TAIL_RE = re.compile(
    rf"(?:\s[-–—]\s{RUBRIC}){{0,2}}(?:\s[-–—])+\s(?:{'|'.join(DASH_SITES)})\s*$"
    r"|\s»\sНовини .*?Завантаження\.\s*$"
    r"|\sПочніть свій день з новини Встановити ZIK\.\s*$"
)


def strip_site_tails(df: pd.DataFrame) -> pd.DataFrame:
    """Cut the site tail and drop the spans that end inside it. Only matching rows are touched."""
    cuts = df.text.map(lambda s: m.start() if (m := SITE_TAIL_RE.search(s)) else -1)
    rows = df.index[cuts >= 0]
    df = df.copy()
    dropped = Counter()
    for col in MARKERS:
        kept = []
        for i in rows:
            spans, cut, text = parse_markers(df.at[i, col]), cuts[i], df.at[i, "text"]
            kept.append(str([s for s in spans if s[1] <= cut]))
            dropped.update((col, text[a:b]) for a, b in spans if b > cut)
        df.loc[rows, col] = kept
    df.loc[rows, "text"] = [df.at[i, "text"][: cuts[i]].rstrip() for i in rows]
    print(f"{len(rows):,} tails cut; spans dropped:", {c: sum(n for (k, _), n in dropped.items() if k == c) for c in MARKERS})
    print("top dropped LOC:", [(s, n) for (k, s), n in dropped.most_common() if k == "loc_markers"][:10])
    return df[df.text.str.len() > 0]


_demo = pd.DataFrame({"text": ["Кабмін у Києві - новини України - Укрaїнa - TCH."],
                      "loc_markers": ["[(9, 14), (34, 40)]"], "org_markers": ["[]"], "per_markers": ["[]"]})
_demo = strip_site_tails(_demo)
assert _demo.text.iloc[0] == "Кабмін у Києві" and _demo.loc_markers.iloc[0] == "[(9, 14)]", _demo.iloc[0].to_dict()
uk_df = strip_site_tails(uk_df)
print(uk_df.text[uk_df.text.str.contains(r"TCH\.$")].head(3).to_list())  # what is left: TCH. inside a real sentence

1 tails cut; spans dropped: {'loc_markers': 1, 'org_markers': 0, 'per_markers': 0}
top dropped LOC: [('Укрaїн', 1)]


14,362 tails cut; spans dropped: {'loc_markers': 5891, 'org_markers': 15488, 'per_markers': 1114}
top dropped LOC: [('Укрaїнa', 1837), ('Світ', 1107), ('Чернівці', 983), ('України', 508), ('Україна', 375), ('Київ', 353), ('Сході', 328), ('Цікaвинки', 102), ('Гламур', 40), ('Києва', 36)]


['Колишня няня з Красноярська, яка в лютому "прославилася" в Росії, потрапивши під суд за побиття дитини, знову потрапила у поле зору поліції, повідомляє TCH.']


### 2.3 Hyphenated place names
The silver labels often cover only part of a hyphenated name, or split it into several spans:

| case | example | uk spans |
|---|---|---|
| split into pieces | `[Івано-][Франківську]`, `[Івано][-][Франківської області]` | 3,425 |
| half without a partner | `[Ріо]-де-Жанейро`, `Ростові-на-[Дону]`, `Івано-[Франківськ]` | 948 |
| other half tagged as PER | `[Тель]-Авіва`, `[Переяслав]-Хмельницькому`, `[Пуерто]-Ріко` | 274 |

The whole hyphenated word is one place, so every LOC span that cuts a hyphenated word grows to the whole word, the
pieces are merged into one span, and a PER span on the same word is dropped.

- **Left as is:** a word where part is tagged ORG (`РБК-[Україна]`, `[Україна]-НАТО`, `[Київ]-Баскет`, 344 spans).
  There the place is a part of an organisation name, which the gold does not label. That needs its own decision.
- The token-level targets of the first two cases barely change, because the tokenizer keeps a hyphenated word as one
  token. What changes is the PER/LOC conflict and the span strings (`Івано` leaves the list of top "locations").
- Applied to uk and ru. Runs before deduplication, so copies whose labels differed only in the split now count as duplicates.
- **Not grown:** a hyphenated word glued to the word before it by a lost space (a lowercase letter right before a
  capital: `вНью-Йорке`, `изСанкт-Петербурга`, 39 ru texts). Growing would put `в` or `из` into the place; 2.5
  drops those texts instead.

In [8]:
WORD_RE = re.compile(r"\w+(?:[-']\w+)*")  # the word part of TOKEN_RE (after apostrophe normalization)
GLUED_RE = re.compile("[а-яёіїєґ][А-ЯЁІЇЄҐ]")  # a lost space: `вНью-Йорке`, `изСанкт-Петербурга`


def overlaps(a: int, b: int, c: int, d: int) -> bool:
    return a < d and c < b


def grow_hyphen_spans(text: str, locs, orgs, pers):
    """Grow every LOC span that cuts a hyphenated word to the whole word (unless ORG covers part of the word),
    merge the overlapping pieces and drop PER spans on those words. Returns (locs, pers), or None if nothing changed."""
    words = [(m.start(), m.end()) for m in WORD_RE.finditer(text) if "-" in m.group()
             and not GLUED_RE.search(m.group())]  # glued words are dropped in 2.5 instead
    grown, fixed = [], []
    for a, b in locs:
        for s, e in words:
            if overlaps(a, b, s, e) and (a > s or b < e) and not any(overlaps(c, d, s, e) for c, d in orgs):
                a, b = min(a, s), max(b, e)
                fixed.append((s, e))
        grown.append((a, b))
    if not fixed:
        return None
    merged = []
    for a, b in sorted(grown):
        if merged and a < merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], b))
        else:
            merged.append((a, b))
    return merged, [(c, d) for c, d in pers if not any(overlaps(c, d, s, e) for s, e in fixed)]


def fix_hyphen_spans(df: pd.DataFrame) -> pd.DataFrame:
    rows = df.index[df.text.str.contains(r"\w-\w") & (df.loc_markers != "[]")]
    df = df.copy()
    changed, new_locs, new_pers, grown = [], [], [], Counter()
    for i in rows:
        text, locs = df.at[i, "text"], parse_markers(df.at[i, "loc_markers"])
        out = grow_hyphen_spans(text, locs, parse_markers(df.at[i, "org_markers"]), parse_markers(df.at[i, "per_markers"]))
        if out is None:
            continue
        changed.append(i)
        new_locs.append(str(out[0]))
        new_pers.append(str(out[1]))
        grown.update(text[a:b] for a, b in set(out[0]) - set(locs))
    per_before = sum(len(parse_markers(v)) for v in df.loc[changed, "per_markers"])
    df.loc[changed, "loc_markers"], df.loc[changed, "per_markers"] = new_locs, new_pers
    per_after = sum(len(parse_markers(v)) for v in new_pers)
    print(f"{len(changed):,} texts relabelled; {per_before - per_after:,} PER spans dropped")
    print("top new LOC spans:", grown.most_common(12))
    return df


_t = "У Тель-Авіві, Івано-Франківській області і РБК-Україна."
assert grow_hyphen_spans(_t, [(2, 6), (14, 19), (19, 20), (20, 40)], [(43, 46)], [(7, 12)]) == ([(2, 12), (14, 40)], [])
assert grow_hyphen_spans(_t, [(47, 54)], [(43, 46)], []) is None  # part of the word is ORG: left as is
uk_df, ru_df = fix_hyphen_spans(uk_df), fix_hyphen_spans(ru_df)

5,565 texts relabelled; 311 PER spans dropped
top new LOC spans: [('Івано-Франківську', 585), ('Івано-Франківська', 446), ('Івано-Франківській області', 373), ('Нью-Йорку', 230), ('Івано-Франківської області', 200), ('Івано-Франківськ', 122), ('Київ-Чоп', 70), ('Нью-Йорка', 54), ('Києво-Печерської лаври', 49), ('Нью-Йорк', 38), ('Київ-Одеса', 36), ('Ріо-де-Жанейро', 30)]


12 texts relabelled; 0 PER spans dropped
top new LOC spans: [('Нью-Йоркe', 1), ('ВНью-Йорке', 1), ('ВКирово-Чепецке', 1), ('ВСанкт-Петербурге', 1), ('Воксхолл-кросc', 1), ('Corot-7b', 1), ('Санкт-Петербург13', 1), ('Санта-Круc', 1), ('Санкт-Петербург8', 1), ('Орехово-Борисово-11ВГ', 1), ('Нью-Йорке8', 1), ('Сент-Джеймcский', 1)]


### 2.4 Punctuation and spaces at span edges
Some LOC spans start or end on punctuation or a space (`вулиці.`, `[Францією, яка ]`, `«Київ`). The tokenizer makes
punctuation its own token, so such a span labels `.` or `«` as part of the place, and the model learns to output
`Київ,`, which the metric counts as wrong. Every LOC span is trimmed to its first and last letter or digit, except a bracket or quote whose partner is inside the span (`России (СКР)`, `аеропорт "Бориспіль"` stay whole); a span
with nothing left is dropped. Applied to uk and ru.

In [9]:
PAIRS = {"(": ")", "«": "»", "“": "”", '"': '"'}  # opener -> closer


def trim_span(text: str, a: int, b: int) -> tuple[int, int]:
    """Strip non-alphanumeric chars from both ends. A bracket or quote stays when its partner is inside the span
    (`России (СКР)`), unless the pair wraps the whole span (`«Київ»` -> `Київ`)."""
    while a < b:
        first, last = text[a], text[b - 1]
        if b - a >= 2 and first in PAIRS and last == PAIRS[first]:
            a, b = a + 1, b - 1
        elif not first.isalnum() and not (first in PAIRS and PAIRS[first] in text[a + 1:b]):
            a += 1
        elif not last.isalnum() and not any(last == c and o in text[a:b - 1] for o, c in PAIRS.items()):
            b -= 1
        else:
            break
    return a, b


def trim_loc_spans(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = df.index[df.loc_markers != "[]"]
    df = df.copy()
    new, changed, examples = [], 0, Counter()
    for i in rows:
        text, spans = df.at[i, "text"], parse_markers(df.at[i, "loc_markers"])
        trimmed = [s for s in (trim_span(text, a, b) for a, b in spans) if s[0] < s[1]]
        if trimmed != spans:
            changed += 1
            examples.update(text[a:b] for a, b in set(spans) - set(trimmed))
        new.append(str(trimmed))
    df.loc[rows, "loc_markers"] = new
    print(f"{name}: {changed:,} texts with trimmed LOC spans; top old spans:", examples.most_common(8))
    return df


_t = 'у «Київ», де России (СКР). аеропорт "Бориспіль".'
_cut = lambda a, b: _t[slice(*trim_span(_t, a, b))]
assert _cut(2, 9) == "Київ" and _cut(13, 26) == "России (СКР)" and _cut(27, 48) == 'аеропорт "Бориспіль"'
assert trim_span("..", 0, 2) == (2, 2)
uk_df, ru_df = trim_loc_spans(uk_df, "uk"), trim_loc_spans(ru_df, "ru")

uk: 2,152 texts with trimmed LOC spans; top old spans: [('"', 59), ('.', 52), ('-', 45), ('вулиці.', 34), ('№3', 20), ('"О', 20), ('вул.', 19), ('№1', 17)]


ru: 462 texts with trimmed LOC spans; top old spans: [('США (', 9), ('России (', 8), ('России)', 3), ('ХМАО-Югра)', 2), ('ЛНР)', 2), ('Южного (', 2), ('Центрального (', 2), ('РФ)', 2)]


### 2.5 Shifted LOC spans
After 2.3 and 2.4, a few LOC spans still end (rarely start) in the middle of a word without a hyphen. They are not
cut words but shifted offsets: the span starts at the right place and runs 5–15 characters too far, usually past a
comma, bracket or quote: `[Київщини, зокре]ма`, `[Харківській (520) обла]сті`, `[Парижа "Шарль де Гол]ль"`,
`[Держдуми Росії від о]купованого`. In the raw data that is 1,295 of 345k uk LOC spans (0.38%) in 1,284 texts, and 76
spans in 68 texts per 1M ru rows. The other spans in those texts are fine. In ru the cause is different: the span is
right but the text lost a space (`в[Шанхае]`, `Президент [Украины]Петр`), so the tokenizer glues the place to the next
word and the BIO target is wrong anyway.

- **Snapping to the word edge** would give `Держдуми Росії від окупованого`, the same garbage but longer.
- **Dropping only the span** leaves a real place (`Київщини`, `Харківській`) tagged `O`, a wrong target of the other
  kind.
- **So the whole text is dropped**, train rows only (about 0.1% of uk). `es` keeps its 10k rows, like in 2.7.
- Words with a hyphen are not checked: 2.3 already grew those spans, and the ones it left (part of the word is ORG,
  `РБК-[Україна]`) are a separate decision.

In [10]:
PLAIN_WORD_RE = re.compile(r"\w+(?:'\w+)*")  # WORD_RE without hyphens: hyphenated words are left to 2.3


def cuts_word(text: str, a: int, b: int) -> bool:
    """True if the span starts or ends strictly inside a word without a hyphen."""
    return any(s < a < e or s < b < e for s, e in (m.span() for m in PLAIN_WORD_RE.finditer(text)))


def drop_shifted_spans(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = df.index[(df.is_valid == 0) & (df.loc_markers != "[]")]
    bad, examples = [], []
    for i in rows:
        text = df.at[i, "text"]
        cut = [text[a:b] for a, b in parse_markers(df.at[i, "loc_markers"]) if cuts_word(text, a, b)]
        if cut:
            bad.append(i)
            examples += cut
    print(f"{name}: {len(bad):,} train texts with a shifted LOC span dropped, e.g.", examples[:6])
    return df.drop(index=bad)


_t = "Біля Києва, Київщини, зокрема Ірпеня; Івано-Франківськ."
assert cuts_word(_t, 12, 27) and not cuts_word(_t, 5, 10) and not cuts_word(_t, 38, 43)  # 38: Івано- left to 2.3
uk_df, ru_df = drop_shifted_spans(uk_df, "uk"), drop_shifted_spans(ru_df, "ru")

uk: 1,288 train texts with a shifted LOC span dropped, e.g. ['Гонконга одра', '41К «м-н Каскад – Онкодисп', 'вул. Варшавську, можуть п', 'Київщини, зокре', 'Києвом стала', 'площі Конституції під В']


ru: 623 train texts with a shifted LOC span dropped, e.g. ['M', 'России', 'Землю', 'Йоркской', 'Ницца', 'Неаполь']


### 2.6 LOC on names of people
The silver labels sometimes tag a person as a place: `Зеленського` is PER 1,489 times and LOC 94 times, `Трампа`
808 / 134, `Порошенка` 459 / 56. A string that the **train** rows tag as PER at least `PER_RATIO` (4) times as often
as LOC, and at least `PER_MIN` (20) times, loses its LOC spans everywhere (train and `es`). The ratio is high on
purpose: `Іраку` is PER 277 / LOC 128 (a country wrongly tagged as PER), and must keep its LOC label. Applied to uk
only: on ru the same rule drops cities that are also common surnames or first names (`Николаев` = Mykolaiv,
`Владимир`, `Виктории`), and Russian test posts do name Ukrainian cities.

In [11]:
PER_RATIO, PER_MIN = 4.0, 20


def drop_person_locs(df: pd.DataFrame, name: str) -> pd.DataFrame:
    train = df[df.is_valid == 0]
    loc_n, per_n = Counter(), Counter()
    for text, locs, pers in zip(train.text, train.loc_markers, train.per_markers):
        if locs != "[]":
            loc_n.update(text[a:b] for a, b in parse_markers(locs))
        if pers != "[]":
            per_n.update(text[a:b] for a, b in parse_markers(pers))
    person = {s for s, n in per_n.items() if s in loc_n and n >= PER_MIN and n >= PER_RATIO * loc_n[s]}
    rows = df.index[df.loc_markers != "[]"]
    df = df.copy()
    new, dropped = [], Counter()
    for i in rows:
        text, spans = df.at[i, "text"], parse_markers(df.at[i, "loc_markers"])
        new.append(str([(a, b) for a, b in spans if text[a:b] not in person]))
        dropped.update(text[a:b] for a, b in spans if text[a:b] in person)
    df.loc[rows, "loc_markers"] = new
    print(f"{name}: {len(person)} person strings, {sum(dropped.values()):,} LOC spans dropped:", dropped.most_common(10))
    return df


uk_df = drop_person_locs(uk_df, "uk")  # not ru: there it hits cities that are also surnames (Николаев, Владимир)

uk: 300 person strings, 1,205 LOC spans dropped: [('Трампа', 133), ('Зеленського', 94), ('Порошенка', 55), ('Тимошенко', 49), ('Януковича', 26), ('Кучми', 24), ('Зорі', 23), ('Луческу', 20), ('Мілана', 18), ('Зеленський', 17)]


### 2.7 Exact duplicates
A duplicate is a row with the same text **and** the same `loc`/`org`/`per` markers. Apostrophe normalization
tail stripping and the hyphen fix run first, so copies that differed only in the apostrophe, the site or
the span split are caught too.
- **uk:** a few thousand copies, no label conflicts. Texts that are also in `es` are dropped from train first, so
  `es` stays clean.
- **ru:** about 6.5% of the rows are copies, mostly news boilerplate (`Об этом сообщает РИА Новости.`). The same
  sentence appears under several `doc_id`s, so removing copies is also what makes a later `doc_id` split leak-free.
- **Label conflicts** (same text, different markers) are not exact duplicates. They are counted here and handled by
  `resolve_label_conflicts`, which drops every copy: there is no way to tell which one is right.

In [12]:
def drop_exact_duplicates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset=["text", *MARKERS])
    print(f"{name}: {before:,} -> {len(df):,} rows ({before - len(df):,} copies dropped)")
    return df


def resolve_label_conflicts(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Texts that are still repeated after exact deduplication have different markers: there is no way to tell
    which copy is right, so every copy is dropped."""
    conflicts = df.text.duplicated(keep=False)
    print(f"{name}: {df.text[conflicts].nunique():,} texts with conflicting labels, {conflicts.sum():,} rows dropped")
    return df[~conflicts]


es_leak = (uk_df.is_valid == 0) & uk_df.text.isin(set(uk_df.text[uk_df.is_valid == 1]))
print(f"uk: {es_leak.sum()} train rows also in es, dropped")
uk_df = resolve_label_conflicts(drop_exact_duplicates(uk_df[~es_leak], "uk"), "uk")
ru_df = resolve_label_conflicts(drop_exact_duplicates(ru_df, "ru"), "ru")
ru_df = ru_df[~ru_df.text.isin(set(uk_df.text))]  # a handful of identical sentences: keep the uk copy
print(uk_df.is_valid.value_counts().rename({0: "train", 1: "es"}).to_dict())

uk: 55 train rows also in es, dropped


uk: 1,008,657 -> 1,006,055 rows (2,602 copies dropped)


uk: 18 texts with conflicting labels, 36 rows dropped


ru: 749,383 -> 735,304 rows (14,079 copies dropped)


ru: 383 texts with conflicting labels, 775 rows dropped


{'train': 996019, 'es': 10000}


### 2.8 ru: boilerplate cap and the final sample
The most repeated ru sentences are agency credits (`Об этом сообщает РИА Новости.`, `... Reuters.`, `Как сообщает
ТАСС, ...`). After deduplication each wording is left once, but there are still thousands of near-identical ones with
no location, and `ТАСС` is sometimes tagged as LOC. They are capped at `BOILERPLATE_CAP` rows. Then `RU_TARGET` rows are
sampled at random (fixed seed).

In [13]:
BOILERPLATE_RE = r"^(?:Об этом|Как|Об этом же) (?:сообща|сообщи|пиш|переда|заяви|информиру|рассказ)"
BOILERPLATE_CAP = 1_000

boiler = ru_df.text.str.contains(BOILERPLATE_RE, regex=True)
print(f"ru: {boiler.sum():,} boilerplate rows, e.g.", ru_df.text[boiler].head(3).to_list())
ru_df = pd.concat([ru_df[~boiler], ru_df[boiler].sample(min(BOILERPLATE_CAP, int(boiler.sum())), random_state=SEED)])
ru_df = ru_df.sample(min(RU_TARGET, len(ru_df)), random_state=SEED)
print(f"ru: {len(ru_df):,} rows kept (target {RU_TARGET:,}), from {ru_df.doc_id.nunique():,} documents")

ru: 20,899 boilerplate rows, e.g. ['Об этом сообщает агентство «Новости-Армения».', 'Об этом рассказало издание Metro.', 'Об этом сообщает Daily Express.']


ru: 250,000 rows kept (target 250,000), from 210,527 documents


## 3. Save
One file, `train_v2.parquet`. The model notebooks read it with `load_corpora_v2` instead of the raw uk CSV; CV and
test still come from the raw CSVs.

In [14]:
TRAIN_V2 = ARTIFACTS / "train_v2.parquet"
train_v2 = pd.concat([uk_df.assign(lang="uk"), ru_df.assign(lang="ru")], ignore_index=True)
train_v2 = train_v2[["text", *MARKERS, "is_valid", "lang"]].astype({"is_valid": "int8"})
train_v2.to_parquet(TRAIN_V2, index=False)

summary = train_v2.assign(split=train_v2.is_valid.map({0: "train", 1: "es"}),
                          has_loc=train_v2.loc_markers != "[]").groupby(["split", "lang"]).agg(
    rows=("text", "size"), with_loc=("has_loc", "mean"))
print(f"wrote {TRAIN_V2} ({TRAIN_V2.stat().st_size / 2**20:.0f} MB)")
summary

wrote /home/taras/Documents/Workspace/KSE/deep_learning/lab-3/artifacts/train_v2.parquet (139 MB)


rows  with_loc
split lang                  
es    uk     10000  0.227200
train ru    250000  0.309860
      uk    996019  0.227479